# notebook used to produce a dataset with bounding box.

We wanted to get bounding boxes for visualization in our streamlit app. 

We had to synchronize our radar data and our images plus the bounding boxes.

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('/home/christophe/ComplexNet/STREAM/labels_CVPR.csv')

records = np.unique(df['dataset'])
print('total number of sequences: ',len(records))
df.columns

total number of sequences:  96


Index(['numSample', 'x1_pix', 'y1_pix', 'x2_pix', 'y2_pix', 'laser_X_m',
       'laser_Y_m', 'radar_X_m', 'radar_Y_m', 'radar_R_m', 'radar_A_deg',
       'radar_D_mps', 'radar_P_db', 'dataset', 'index', 'Annotation',
       'Difficult'],
      dtype='object')

In [21]:
sequence = 'RECORD@2020-11-21_13.57.07'
#sequence='RECORD@2020-11-21_11.54.31'
root_folder=f'/home/christophe/RADIalP7/DATASET/{sequence}'
dataset=sequence

sub_df=df.loc[df['dataset'] == dataset]
print(f'labels found for sequence: {sequence}: {sub_df.shape[0]} ')

strong_df=df.loc[(df['Annotation'] == 'strong') & (df['dataset'] == dataset)]
print(f'qualitative labels found for sequence: {sequence}: {strong_df.shape[0]} ')

labels found for sequence: RECORD@2020-11-21_13.57.07: 211 
qualitative labels found for sequence: RECORD@2020-11-21_13.57.07: 40 


In [15]:
sub_df.to_csv(f'stream_labels_{sequence}.csv',index=False)

# GITHUB EXPLORATION

In [ ]:
from DBReader.DBReader import SyncReader
from custom_signal_process  import RadarSignalP7
import os
import pandas as pd
import numpy as np
import sys
calib_path='/home/christophe/RADIalP7/SignalProcessing/CalibrationTable.npy'
RSP = RadarSignalP7(path_calib_mat=calib_path,method='RD',device='cpu')
labels = pd.read_csv('/home/christophe/ComplexNet/STREAM/labels_CVPR.csv')


records = ['RECORD@2020-11-21_13.57.07']

data_dir='/home/christophe/RADIalP7/DATASET'

save_folder='/home/christophe/RADIalP7/STREAM2/'

adc_folder=save_folder+'/ADC/'
fft_folder=save_folder+'/FFT/'
fft2_folder=save_folder+'/FFT2/'
image_folder=save_folder+'/IMG'
label_folder=save_folder+'/LABELS'
SAVE_ADC=True
SAVE_IMG=True
SAVE_RANGE_FFT=True
SAVE_RD=True
save_labels=True


if not os.path.exists(save_folder):
    os.makedirs(save_folder)
    os.makedirs(adc_folder)
    os.makedirs(fft_folder)
    os.makedirs(fft2_folder)
    os.makedirs(image_folder)
    os.makedirs(label_folder)
    print('succesfully created folders where data will be saved')
else:
    if SAVE_ADC or SAVE_IMG or SAVE_RANGE_FFT or SAVE_RD  or save_labels:
        sys.exit('Warning sequence seems to have been already computed')
num_samples_list = []
for i,record in enumerate(records):
    print(i,". ",record)
    boxes = labels[labels.dataset == record] 
    
    root_folder = os.path.join(data_dir,record)
    db = SyncReader(root_folder,tolerance=20000,silent=True)
    db_table=db.table
    
    print('debug type of db table: ',type(db_table))

    unique_indices = np.unique(boxes['index'])
  
    count=0
    for index in unique_indices:
            print('debug index: ',index)
            labels_df=boxes[boxes['index'] == index]
            if labels_df.shape[0]>0:
                save_label_path=os.path.join(label_folder,f'bboxes_{index}.csv')
                labels_df.to_csv(save_label_path,index=False)
            numSample = boxes[boxes['index'] == index]['numSample'].iloc[0]

            #print('index and numSample: ',index,numSample)
            num_samples_list.append((index,numSample))
            #continue
            
            sample = db.GetSensorData(index)

            image=sample['camera']['data']
            
            if SAVE_IMG:
                save_img_path=os.path.join(image_folder,f'img_{index}.npy')
                np.save(save_img_path,arr=image)


            raw_adc=RSP.run(sample['radar_ch0']['data'],sample['radar_ch1']['data'],sample['radar_ch2']['data'],sample['radar_ch3']['data'])
            save_adc_path=os.path.join(adc_folder,f'raw_adc_{index}.npy')
            if SAVE_ADC:
                np.save(save_adc_path,raw_adc)
            first_fft=RSP.get_first_fft(sample['radar_ch0']['data'],sample['radar_ch1']['data'],sample['radar_ch2']['data'],sample['radar_ch3']['data'])
            save_fft_path=os.path.join(fft_folder,f'first_fft_{index}.npy')
            if SAVE_RANGE_FFT:
                np.save(save_fft_path,first_fft)

            second_fft=RSP.compute_second_fft(first_fft)
            save_path2=os.path.join(fft2_folder,f'second_fft_{index}.npy')


            if SAVE_RD:
                np.save(save_path2,second_fft)
            
            count+=1
            #filename = os.path.join(config['Output_Folder'],"adc_{:06d}".format(numSample))
            #np.save(filename,adc) 
print('parsed: ',count,'samples')
duplicate_num_samples = {x for x in num_samples_list if num_samples_list.count(x) > 1}

unique_num_samples = {x for x in num_samples_list if num_samples_list.count(x) ==1}

# Print the duplicates
if duplicate_num_samples:
    print('Duplicate numSample values found:', duplicate_num_samples)
else:
    print('No duplicate numSample values found.')


CPU will be used to execute the processing
succesfully created folders where data will be saved
0 .  RECORD@2020-11-21_13.57.07
debug type of db table:  <class 'numpy.ndarray'>
NIGHTLY TEST
debug index:  0
debug index:  1
debug index:  2
debug index:  4
debug index:  5
debug index:  6
debug index:  7
debug index:  11
debug index:  13
debug index:  14
debug index:  16
debug index:  17
debug index:  18
debug index:  19
debug index:  20
debug index:  21
debug index:  22
debug index:  23
debug index:  24
debug index:  25
debug index:  26
debug index:  27
debug index:  28
debug index:  30
debug index:  31
debug index:  32
debug index:  33
debug index:  36
debug index:  37
debug index:  38
debug index:  39
debug index:  40
debug index:  41
debug index:  43
debug index:  44
debug index:  45
debug index:  46
debug index:  50
debug index:  51
debug index:  52
debug index:  53
debug index:  54
debug index:  55
debug index:  56
debug index:  57
debug index:  58
debug index:  59
debug index:  60
d